# Re-export the turbine model — no retraining

The first export int8-quantised the whole graph. Measured on the same test split that
produced `best.pt`'s numbers:

| | mAP50 | mAP50-95 |
|---|---|---|
| `best.pt` | 0.759 | **0.478** |
| int8 everywhere | 0.667 | **0.239** |

mAP50 fell 12%; mAP50-95 fell 50%. mAP50 only asks whether a box overlaps the truth at
all, while mAP50-95 averages over tight IoU thresholds — so the model still finds defects
but places the boxes loosely. That is what quantising the detection head's coordinate
regression to 8 bits does. 25 of the 88 quantised convolutions sit in that head.

This notebook exports three variants, evaluates each on the same split, and keeps the
smallest one that stays within 5% of full precision. **No GPU needed and no retraining** —
it reuses the `best.pt` you already trained.

### Setup
1. **Add Input → Your Work → Notebook Output →** the `turbine_v2_kaggle` version that
   finished training. That is where `best.pt` lives.
2. **Internet: On** (to clone the repo).
3. Accelerator can stay **None** — evaluation runs on CPU, so this costs no GPU quota.
4. **Save Version → Save & Run All (Commit)**, as always.

In [ ]:
!pip install -q ultralytics onnx onnxruntime

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/abyyworld/Drone-visualisation-training.git"
BRANCH   = "claude/model-retrain-solar-panels-n77z3v"
WORK     = Path("/kaggle/working/drone-inspection")

if not WORK.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPO_URL, str(WORK)], check=True)
os.chdir(WORK); sys.path.insert(0, str(WORK))
print("repo ->", WORK)

# Find best.pt in whatever was attached, without assuming the input folder's name.
found = sorted(Path("/kaggle/input").rglob("weights/best.pt")) if Path("/kaggle/input").exists() else []
assert found, (
    "No best.pt found under /kaggle/input.\n"
    "Add Input -> Your Work -> Notebook Output -> the finished turbine_v2_kaggle version."
)
BEST = found[0]
print("weights ->", BEST, f"({BEST.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# The test split must be the one the model was measured on. rebuild_turbine.py seeds its
# RNG per split, so rebuilding reproduces the identical split rather than a fresh shuffle.
subprocess.run([
    "python3", "tools/rebuild_turbine.py",
    "--out", "/kaggle/working/turbine_v2",
    "--copy", "--min-sharpness", "20", "--keep-augmented",
], check=True)

In [ ]:
subprocess.run([
    "python3", "tools/export_variants.py", str(BEST),
    "--data", "/kaggle/working/turbine_v2/data.yaml",
    "--name", "turbine", "--imgsz", "960",
], check=True)

## Publish

Commits the winning export and the comparison table to the working branch. Pushes to the
branch, never to `main`, so the live site is unchanged until you merge.

In [ ]:
subprocess.run([
    "python3", "tools/publish_results.py", "--name", "turbine",
], check=True)